# This Notebook

Makes use of OpenMed/OpenMed-NER-* token classifer models, spaCy, medspacy and detects medical entities like diseases and medicines in text.

- The OpenMed models provide possible token labels for entity detection. We build a custom spaCy pipeline layer on top of them.
- spaCy is used for building the NER pipeline.
- Medspacy only provides the sentecizer and context processor. Medspacy by itself can't detect entities, that's why the models are needed.

**TODO**: Deduplicate entities. For instance, "Acquired immunodeficiency syndrome" can be also be written as "AIDS". If both representation are used in a text, make them boil down boil down to one, or "coreference" them.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from dataclasses import dataclass

import spacy
from spacy.language import Language
from spacy.tokens import Doc, Span

import medspacy # even though unused, this needed to function
from loguru import logger

from medspacy.visualization import visualize_ent

## Custom Token Extractor based on OpenMed models

In [2]:
@dataclass
class Entity:
    text:  str
    score: float
    start: int
    end:   int

class EntityExtractor:
    tokenizer: AutoTokenizer
    model:     AutoModelForTokenClassification

    def __init__(self, classifier_key: str, device_map: str | None = None) -> None:
        tokenizer = AutoTokenizer.from_pretrained(classifier_key, use_fast = True)
        model     = AutoModelForTokenClassification.from_pretrained(classifier_key, device_map = device_map)
        model.eval()

        self.model = model
        self.tokenizer = tokenizer

    def __call__(self, text: str):
        encoded_tokens = self.tokenizer(text, return_tensors = 'pt', return_offsets_mapping = True, truncation = True) # type: ignore
        token_spans    = encoded_tokens.pop('offset_mapping')[0]
        token_count    = encoded_tokens['input_ids'].shape[1]

        with torch.no_grad():
            outputs = self.model(**encoded_tokens) # type: ignore

        logits     = outputs.logits[0]
        pred_ids   = logits.argmax(dim = -1)
        pred_probs = logits.softmax(dim = -1)
        tokens     = self.tokenizer.convert_ids_to_tokens(encoded_tokens['input_ids'][0]) # type: ignore

        # filter pass
        subword_entities: list[tuple[Entity, int]] = []
        for i in range(token_count):
            toktype = pred_ids[i]
            if toktype == 0:
                continue
            subword_entities.append(
                (
                    Entity(
                        text  = tokens[i],
                        score = pred_probs[i, toktype].item(),
                        start = token_spans[i, 0].item(),
                        end   = token_spans[i, 1].item()
                    ),
                    toktype.item()
                )
            )


        # stitch pass
        stitched_entities: list[tuple[Entity, int]] = []
        stitch_buffer: list[tuple[Entity, int]] = []
        def stitch():
            nonlocal stitch_buffer
            if len(stitch_buffer) == 0:
                return
            stitched_entities.append(
                (
                    Entity(
                        text  = ''.join(map(lambda item : item[0].text.replace('##', ''), stitch_buffer)),
                        score = stitch_buffer[0][0].score,
                        start = stitch_buffer[0][0].start,
                        end   = stitch_buffer[-1][0].end,
                    ),
                    stitch_buffer[0][1]
                )
            )
            stitch_buffer = []

        i = 0
        while i < len(subword_entities):
            assert not subword_entities[i][0].text.startswith('##'), 'Token at this index should not start with ##'
            stitch_buffer.append(subword_entities[i])
            j = i + 1
            increment = 1
            while j < len(subword_entities) and subword_entities[j][0].text.startswith('##'):
                stitch_buffer.append(subword_entities[j])
                j += 1
                increment += 1
            stitch()
            i += increment
        stitch()


        # word chaining pass
        chained_entities: list[Entity] = []
        chain_buffer: list[Entity] = []
        def chain():
            nonlocal chain_buffer
            if len(chain_buffer) == 0:
                return
            chained_entities.append(
                Entity(
                    # text  = ' '.join(map(lambda item : item.text, chain_buffer)),
                    text  = text[chain_buffer[0].start:chain_buffer[-1].end],
                    score = chain_buffer[0].score,
                    start = chain_buffer[0].start,
                    end   = chain_buffer[-1].end,
                )
            )
            chain_buffer = []

        i = 0
        while i < len(stitched_entities):
            assert stitched_entities[i][1] != 2, 'Sub-entity at this index should not be of type 2'
            chain_buffer.append(stitched_entities[i][0])
            j = i + 1
            increment = 1
            while j < len(stitched_entities) and stitched_entities[j][1] == 2:
                chain_buffer.append(stitched_entities[j][0])
                j += 1
                increment += 1
            chain()
            i += increment
        chain()

        return chained_entities

## Custom spaCy pipeline that utilizes the extractors

In [3]:
@Language.factory('medical_ner')
class CustomNER:
    disease_extractor:  EntityExtractor
    medicine_extractor: EntityExtractor

    def __init__(self, nlp: Language, name: str):
        self.disease_extractor  = EntityExtractor('OpenMed/OpenMed-NER-DiseaseDetect-BioClinical-108M')
        self.medicine_extractor = EntityExtractor('OpenMed/OpenMed-NER-PharmaDetect-BioPatient-108M')
        pass

    def merge_entity_sequences(self, sequences: list[list[Entity]], types: list[str]):
        assert len(sequences) == len(types), "Length of sequence list and type list must match"

        merged_sequence: list[tuple[Entity, str]] = []

        for sequence, seq_type in zip(sequences, types):
            merged_sequence.extend(zip(
                sequence,
                [ seq_type ] * len(sequence)
            ))

        merged_sequence.sort(key = lambda e_t : e_t[0].start)

        return merged_sequence

    def __call__(self, doc: Doc):
        diseases  = self.disease_extractor(doc.text)
        medicines = self.medicine_extractor(doc.text)

        spacy_spans: list[Span] = []

        for entity, entype in self.merge_entity_sequences([ medicines, diseases ], [ 'MEDICINE', 'DISEASE' ]):
            span = doc.char_span(entity.start, entity.end, label = entype)
            if span is not None:
                spacy_spans.append(span)

        doc.ents = spacy_spans

        return doc

## Build pipeline

In [4]:
nlp = spacy.blank("en")
nlp.add_pipe('medspacy_pyrush')  # from medspacy
nlp.add_pipe('medical_ner')      # our solution
nlp.add_pipe('medspacy_context') # from medspacy

logger.remove() # Make medspacy PyRUSH silent, otherwise logs too much.

W0515 20:13:41.006000 26328 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Test

In [5]:
TEXT = """
**Clinical Progress Note**
**Patient:** M.T., 72-year-old female  

**Confirmed diagnoses:**  
- Hypertension (stage 2)  
- Osteoarthritis (bilateral knees)  

**Uncertain diagnoses (to be ruled out):**
- Possible early-stage Parkinson's disease (mild resting tremor noted, follow-up in 3 months)
- Suspected medication-induced gastritis (endoscopy pending)  

**Negated diagnoses (explicitly ruled out or denied):**  
- No history of myocardial infarction or stroke  
- Denies any prior diagnosis of chronic kidney disease  
- No evidence of diabetic retinopathy on recent exam  

**Prior medications (patient has used):**  
- Lisinopril 20 mg/day (discontinued due to chronic cough)  
- Ibuprofen 400 mg as needed (stopped due to dyspepsia)  

**Medications to be prescribed (starting today):**  
- Losartan 50 mg/day  
- Acetaminophen 500 mg as needed for joint pain  
- Pantoprazole 40 mg/day for 4 weeks pending gastritis workup
"""

doc = nlp(TEXT)

for ent in doc.ents:
    print(f'[{ent.label_}@{ent.start_char}:{ent.end_char}] {ent.text}')

[DISEASE@98:110] Hypertension
[DISEASE@125:139] Osteoarthritis
[DISEASE@227:246] Parkinson's disease
[DISEASE@261:267] tremor
[DISEASE@329:338] gastritis
[DISEASE@436:467] myocardial infarction or stroke
[DISEASE@502:524] chronic kidney disease
[DISEASE@544:564] diabetic retinopathy
[MEDICINE@629:639] Lisinopril
[DISEASE@671:684] chronic cough
[MEDICINE@690:699] Ibuprofen
[DISEASE@733:742] dyspepsia
[MEDICINE@802:810] Losartan
[MEDICINE@825:838] Acetaminophen
[DISEASE@860:870] joint pain
[MEDICINE@875:887] Pantoprazole
[DISEASE@918:927] gastritis


## Visualize

In [7]:
visualize_ent(doc)